In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/stranger-things/Stranger.csv


In [4]:
print("Project Upside Down")
print("Predicting Stranger things Episode Rating")

Project Upside Down
Predicting Stranger things Episode Rating


In [11]:
# ██████████████████████████████████████████████████████████
#   STRANGER THINGS × MACHINE LEARNING
#   "Project Upside Down" – Predicting Stranger things Episode Rating
# ██████████████████████████████████████████████████████████

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Christmas lights ON
plt.style.use('dark_background')
sns.set_palette("dark:red") 

print("HAWKINS NATIONAL LABORATORY TERMINAL v1985")
print("Loading Stranger.csv from your Kaggle dataset...")

HAWKINS NATIONAL LABORATORY TERMINAL v1985
Loading Stranger.csv from your Kaggle dataset...


In [25]:
df = pd.read_csv("/kaggle/input/d/adar5hsharma/stranger-things/Stranger.csv")

# Quick fixes for the typos we know exist
df.loc[(df['season']==1) & (df['episode']==4), 'votes'] = 31000
df.loc[(df['season']==1) & (df['episode']==4), 'rating'] = 8.9

df['title'] = df['title'].str.title().str.strip()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
df['previous_ep_rate'] = pd.to_numeric(df['previous_ep_rate'], errors='coerce')
df['release_date'] = pd.to_datetime(df['release_date'], dayfirst=True, errors='coerce')

# Boolean columns
df['is_finale'] = df['is_finale'].astype(str).str.upper() == 'TRUE'
df['season_finale'] = df['final_ep'].astype(str).str.upper() == 'TRUE'

print(f"Loaded {len(df)} episodes → {df['rating'].notna().sum()} have ratings")
df.head()
df.info()

Loaded 42 episodes → 38 have ratings
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   season            42 non-null     int64         
 1   episode           42 non-null     int64         
 2   title             42 non-null     object        
 3   rating            38 non-null     float64       
 4   votes             38 non-null     float64       
 5   is_finale         42 non-null     bool          
 6   season_avg        38 non-null     float64       
 7   final_ep          42 non-null     bool          
 8   release_date      42 non-null     datetime64[ns]
 9   ep_runtime        42 non-null     int64         
 10  previous_ep_rate  34 non-null     float64       
 11  season_finale     42 non-null     bool          
dtypes: bool(3), datetime64[ns](1), float64(4), int64(3), object(1)
memory usage: 3.2+ KB


In [19]:
data = df.dropna(subset=['rating']).copy().reset_index(drop=True)

data['episode_in_season'] = data['episode']
data['is_premiere'] = (data['episode'] == 1).astype(int)
data['runtime_hours'] = data['ep_runtime'] / 60
data['prev_rating'] = data['previous_ep_rate'].fillna(method='bfill').fillna(8.5)
data['log_votes'] = np.log1p(data['votes'])

# Days between episodes
data = data.sort_values(['season','episode']).reset_index(drop=True)
data['days_since_prev'] = data['release_date'].diff().dt.days.fillna(0)

print("Feature engineering complete. The lights are blinking!")

Feature engineering complete. The lights are blinking!


In [20]:
features = ['season','episode_in_season','is_premiere','is_finale',
            'season_finale','runtime_hours','prev_rating',
            'days_since_prev','log_votes']

X = data[features]
y = data['rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11)

model = RandomForestRegressor(n_estimators=500, random_state=85, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f"Demogorgon Fear Level (MAE) : ±{mean_absolute_error(y_test,preds):.3f}")
print(f"Christmas Lights Strength (R²) : {r2_score(y_test,preds):.3f}")

Demogorgon Fear Level (MAE) : ±0.333
Christmas Lights Strength (R²) : 0.630


In [24]:
print("One last Adventure")
print("\nOPENING THE FINAL GATE – SEASON 5 VOLUME 2 PREDICTIONS\n")

future = df[df['rating'].isna()].copy()
future['episode_in_season'] = future['episode']
future['is_premiere'] = 0
future['runtime_hours'] = future['ep_runtime'] / 60
future['prev_rating'] = future['previous_ep_rate'].fillna(9.4)
future['days_since_prev'] = [30, 14, 6, 6]  
future['log_votes'] = np.log1p(35000)

X_future = future[features].fillna(0)
future['predicted_rating'] = model.predict(X_future)

print("PREDICTED RATINGS")
print("="*55)
for _, row in future.iterrows():
    finale = " SERIES FINALE!!!" if row['season_finale'] else ""
    print(f"S{row['season']}E{row['episode']:02d} │ {row['title']:<30} │ {row['predicted_rating']:.2f} ⭐{finale}")

print("\nFriends don’t lie. The Mind Flayer has spoken.")

One last Adventure

OPENING THE FINAL GATE – SEASON 5 VOLUME 2 PREDICTIONS

PREDICTED RATINGS
S5E05 │ Shock Jock                     │ 9.05 ⭐
S5E06 │ Escape From Camazotz           │ 9.06 ⭐
S5E07 │ The Bridge                     │ 9.15 ⭐
S5E08 │ The Rightside Up               │ 9.19 ⭐ SERIES FINALE!!!

Friends don’t lie. The Mind Flayer has spoken.
